# 🧠 Shap-E Text/Image to 3D Generator with Gradio UI

In [ ]:
!git clone https://github.com/openai/shap-e.git
%cd shap-e
!pip install -e .
!pip install -U gradio trimesh


In [ ]:
import torch
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images, decode_latent_mesh
from shap_e.util.image_util import load_image
import gradio as gr
import trimesh
from PIL import Image as PILImage
import numpy as np
import os
import uuid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load models
xm = load_model('transmitter', device=device)
model_text = load_model('text300M', device=device)
model_image = load_model('image300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))


In [ ]:
def generate_3d(text_prompt, uploaded_image):
    input_type = "text" if text_prompt else "image"
    batch_size = 1
    guidance_scale = 15.0 if input_type == "text" else 3.0

    if input_type == "text":
        latents = sample_latents(
            batch_size=batch_size,
            model=model_text,
            diffusion=diffusion,
            guidance_scale=guidance_scale,
            model_kwargs=dict(texts=[text_prompt]),
            progress=True,
            clip_denoised=True,
            use_fp16=True,
            use_karras=True,
            karras_steps=64,
            sigma_min=1e-3,
            sigma_max=160,
            s_churn=0,
        )
    else:
        pil_image = PILImage.fromarray(uploaded_image.astype("uint8"))
        image = load_image(pil_image)
        latents = sample_latents(
            batch_size=batch_size,
            model=model_image,
            diffusion=diffusion,
            guidance_scale=guidance_scale,
            model_kwargs=dict(images=[image]),
            progress=True,
            clip_denoised=True,
            use_fp16=True,
            use_karras=True,
            karras_steps=64,
            sigma_min=1e-3,
            sigma_max=160,
            s_churn=0,
        )

    latent = latents[0]
    mesh = decode_latent_mesh(xm, latent).tri_mesh()

    uid = str(uuid.uuid4())[:8]
    obj_path = f"/content/model_{uid}.obj"
    stl_path = f"/content/model_{uid}.stl"
    gif_path = f"/content/render_{uid}.gif"

    with open(obj_path, "w") as f:
        mesh.write_obj(f)

    tri_mesh = trimesh.Trimesh(vertices=mesh.verts, faces=mesh.faces)
    tri_mesh.export(stl_path)

    cameras = create_pan_cameras(64, device)
    images = decode_latent_images(xm, latent, cameras, rendering_mode="nerf")
    images[0].save(gif_path, save_all=True, append_images=images[1:], duration=100, loop=0)

    return gif_path, obj_path, stl_path


In [ ]:
iface = gr.Interface(
    fn=generate_3d,
    inputs=[
        gr.Textbox(label="Enter a Text Prompt (or leave blank if uploading image)"),
        gr.Image(type="numpy", label="Or Upload an Image"),
    ],
    outputs=[
        gr.Image(type="filepath", label="Generated 3D GIF (rotating view)"),
        gr.File(label="Download OBJ"),
        gr.File(label="Download STL")
    ],
    title="Text or Image to 3D Generator with Shap-E",
    description="Generate a 3D model (.obj/.stl) and a preview GIF using a text prompt or an image input."
)

iface.launch(share=True, allowed_paths=["/content"])
